# This code is a demonstration on model usage
The model used was packaged using 'pickle'

In [1]:
# Importing Libraries
import pandas as pd
import pickle
import numpy as np
import sys
import os

In [3]:
# Defining Features function
def features(df, mapping):
    ft_df = df.copy()

    # Defining categories from the CSV
    cats = mapping['feature_type'].unique()

    for cat in cats:
        categories = mapping[mapping['feature_type'] == cat]['feature_name'].tolist()
        # Identifying features that survived clean
        ex_features = [f for f in categories if f in ft_df.columns]

        if ex_features:
            ft_df[f'total_{cat.split("_")[0]}_intensity'] = ft_df[ex_features].sum(axis=1, min_count=1)
    return ft_df


# Loading model
def run_test(folder='extracted_monthly_data'):
    print("Test Script:")
    try:
        with open('xgboost_model.pkl', 'rb') as f:
            model = pickle.load(f)
        mapping = pd.read_csv('feature_name_to_number_mapping.csv')
        f_dict = dict(zip(mapping['feature_number'], mapping['feature_name']))
        print("Loaded model and mapped features")
    except Exception as e:
        print(f"Error loading files: {e}")
        return

    # Processing data folder
    print(f"Processing files in: {folder}")
    data = []
    
    if not os.path.exists(folder):
        print(f"Error: Folder '{folder}' not found.")
        return

    count = 0
    for filename in os.listdir(folder):
        if count >= 1000: break # Limiting Sample Size
        if filename.endswith(".txt"): # Loop iterating through all files 
            path = os.path.join(folder, filename)
            with open(path, 'r') as f:
                for line in f:
                    if count >= 1000: break # Stops once limit is reached
                    strip = line.strip().split()
                    if not strip: continue
                    
                    # Row setup
                    row = {'source_file': filename}
                    # Parsing the index
                    for item in strip[1:]:
                        f_num, f_val = item.split(':')
                        f_name = f_dict.get(int(f_num), f"feat_{f_num}")
                        row[f_name] = float(f_val)
                    data.append(row)
                    count += 1 # Increment record count

    raw_df = pd.DataFrame(data).drop_duplicates()
    f_df = features(raw_df, mapping)
    # Match the model features
    model_features = model.get_booster().feature_names
    df = f_df.reindex(columns=model_features, fill_value=np.nan)

    print(f"Model runs on {len(df)} records")
    probs = model.predict_proba(df)[:, 1]
    preds = (probs >= 0.3).astype(int)

    # Output results
    results = pd.DataFrame({
        'Source_File': raw_df['source_file'],
        'Risk_Score': probs,
        'Label': preds
    })
    
    results.to_csv('test_results.csv', index=False) # Saving result to csv
    print("Test Complete")
    print("Results saved to 'test_results.csv'")

if __name__ == "__main__":
    path = sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith('-') else 'extracted_monthly_data'
    run_test(path)

Test Script:
Loaded model and mapped features
Processing files in: extracted_monthly_data
Model runs on 671 records
Test Complete
Results saved to 'test_results.csv'
